# SVM Baseline for IMDB Sentiment Classification

This notebook demonstrates how to create an SVM-based baseline for binary sentiment classification on the IMDB Large Movie Review dataset. We'll reuse the existing data loading and preprocessing utilities, and compare the SVM performance to the existing Logistic Regression baseline.

## Overview

- **Dataset**: IMDB Large Movie Review Dataset (positive vs negative reviews)
- **Features**: TF-IDF vectorization (same as Logistic Regression baseline)
- **Classifier**: LinearSVC (efficient for sparse high-dimensional data)
- **Comparison**: Direct comparison with Logistic Regression on identical features

## 1. Import Libraries

Import the necessary libraries for data loading, preprocessing, modeling, and evaluation.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
from src.data.imdb_loader import load_imdb_dataset
from src.preprocessing.text_preprocessing import preprocess_texts

# Import scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Import visualization and utilities
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

## 2. Load and Preprocess Data

Load the IMDB dataset and apply the same preprocessing used by the Logistic Regression baseline.

In [ ]:
# Load IMDB dataset
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "aclImdb"

print(f"Loading IMDB data from: {DATA_DIR}")
train_texts, train_labels, test_texts, test_labels = load_imdb_dataset(DATA_DIR)

print(f"Training samples: {len(train_texts)}")
print(f"Test samples: {len(test_texts)}")
print(f"Label distribution - Train: {np.bincount(train_labels)}")
print(f"Label distribution - Test: {np.bincount(test_labels)}")

# Preprocess texts (same as Logistic Regression baseline)
MAX_LENGTH = 250  # Limit document length for efficiency
print(f"\nPreprocessing texts with max_length={MAX_LENGTH}...")

train_texts_processed = preprocess_texts(train_texts, max_length=MAX_LENGTH)
test_texts_processed = preprocess_texts(test_texts, max_length=MAX_LENGTH)

print("Preprocessing complete!")
print(f"Sample processed text: {train_texts_processed[0][:100]}...")

## 3. Define SVM Pipeline

Create a scikit-learn Pipeline that combines TF-IDF vectorization with LinearSVC classifier. We'll use the same vectorizer settings as the Logistic Regression baseline for fair comparison.

In [ ]:
# Define TF-IDF + SVM pipeline
# Using same vectorizer settings as Logistic Regression baseline
tfidf_params = {
    'lowercase': True,
    'ngram_range': (1, 2),  # unigrams and bigrams
    'min_df': 3,
    'max_df': 0.95,
    'max_features': 30000,
    'smooth_idf': True,
    'use_idf': True,
}

svm_params = {
    'C': 1.0,  # Regularization parameter
    'max_iter': 2000,
    'random_state': 42,
    'dual': False,  # More efficient for n_samples > n_features
}

# Create the pipeline
svm_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(**tfidf_params)),
    ('classifier', LinearSVC(**svm_params))
])

print("SVM Pipeline created:")
print(f"- Vectorizer: TfidfVectorizer with ngram_range={tfidf_params['ngram_range']}, max_features={tfidf_params['max_features']}")
print(f"- Classifier: LinearSVC with C={svm_params['C']}, max_iter={svm_params['max_iter']}")

## 4. Train the SVM Model

Train the SVM pipeline on the preprocessed training data.

In [ ]:
print("Training SVM model...")
print("(This may take a few minutes depending on your hardware)")

# Train the pipeline
svm_pipeline.fit(train_texts_processed, train_labels)

print("Training complete!")
print(f"Number of features: {len(svm_pipeline.named_steps['vectorizer'].get_feature_names_out())}")

## 5. Evaluate the SVM Model

Compute accuracy, precision, recall, and F1 score on the test set.

In [ ]:
# Make predictions on test set
print("Evaluating SVM model on test set...")
y_pred_svm = svm_pipeline.predict(test_texts_processed)

# Compute metrics
svm_metrics = {
    'accuracy': accuracy_score(test_labels, y_pred_svm),
    'precision': precision_score(test_labels, y_pred_svm),
    'recall': recall_score(test_labels, y_pred_svm),
    'f1': f1_score(test_labels, y_pred_svm)
}

print("\n=== SVM Test Results ===")
print(f"Accuracy:  {svm_metrics['accuracy']:.4f}")
print(f"Precision: {svm_metrics['precision']:.4f}")
print(f"Recall:    {svm_metrics['recall']:.4f}")
print(f"F1 Score:  {svm_metrics['f1']:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(test_labels, y_pred_svm, target_names=['Negative', 'Positive']))

## 6. Compare to Logistic Regression

Load the Logistic Regression baseline results and compare performance on the same TF-IDF features.

In [ ]:
import json
from pathlib import Path

# Load Logistic Regression results
REPORTS_DIR = PROJECT_ROOT / "reports"
lr_results_files = list(REPORTS_DIR.glob("best_baseline_results_*.json"))

if lr_results_files:
    # Get the most recent LR results file
    latest_lr_file = max(lr_results_files, key=lambda f: f.stat().st_mtime)
    print(f"Loading LR results from: {latest_lr_file}")

    with open(latest_lr_file, 'r') as f:
        lr_results = json.load(f)

    # Extract TF-IDF results
    lr_tfidf = lr_results.get('tfidf', {})
    if lr_tfidf:
        lr_metrics = lr_tfidf['metrics']

        print("\n" + "="*60)
        print("PERFORMANCE COMPARISON: SVM vs Logistic Regression")
        print("="*60)

        # Create comparison table
        comparison_data = {
            'Model': ['Logistic Regression', 'SVM (LinearSVC)'],
            'Accuracy': [lr_metrics['accuracy'], svm_metrics['accuracy']],
            'Precision': [lr_metrics['precision'], svm_metrics['precision']],
            'Recall': [lr_metrics['recall'], svm_metrics['recall']],
            'F1 Score': [lr_metrics['f1'], svm_metrics['f1']]
        }

        df_comparison = pd.DataFrame(comparison_data)
        print(df_comparison.to_string(index=False, float_format='%.4f'))

        # Calculate differences
        print("\nDifferences (SVM - LR):")
        for metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score']:
            diff = svm_metrics[metric.lower()] - lr_metrics[metric.lower()]
            print(f"{metric}: {diff:+.4f}")

        print(f"\nLR Configuration: {lr_tfidf.get('config_name', 'Unknown')}")
        print("SVM Configuration: TF-IDF unigrams+bigrams + LinearSVC"
    else:
        print("No TF-IDF results found in LR baseline file")
else:
    print("No Logistic Regression baseline results found.")
    print("Run the LR baseline first to enable comparison.")

## 7. Confusion Matrix Visualization

Plot a confusion matrix to visualize the SVM model's predictions.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(test_labels, y_pred_svm)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
ax1.set_title('Confusion Matrix (Counts)')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues', ax=ax2,
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
ax2.set_title('Confusion Matrix (Normalized)')
ax2.set_ylabel('True Label')
ax2.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("Confusion Matrix Interpretation:")
print(f"True Negatives: {cm[0,0]} reviews correctly classified as negative")
print(f"False Positives: {cm[0,1]} negative reviews misclassified as positive")
print(f"False Negatives: {cm[1,0]} positive reviews misclassified as negative")
print(f"True Positives: {cm[1,1]} reviews correctly classified as positive")

## Summary

This notebook demonstrated how to create an SVM-based baseline for IMDB sentiment classification:

1. **Data Loading**: Reused existing utilities to load and preprocess IMDB reviews
2. **Pipeline Creation**: Built a TF-IDF + LinearSVC pipeline with the same vectorizer settings as Logistic Regression
3. **Training**: Trained the SVM model on the preprocessed training data
4. **Evaluation**: Computed standard classification metrics (accuracy, precision, recall, F1)
5. **Comparison**: Compared SVM performance to the existing Logistic Regression baseline
6. **Visualization**: Plotted confusion matrices for detailed error analysis

### Key Findings
- SVM with LinearSVC provides an alternative to Logistic Regression for text classification
- Both models use the same TF-IDF features, making direct comparison possible
- LinearSVC is efficient for sparse high-dimensional data like TF-IDF vectors

### Next Steps
- Experiment with different SVM hyperparameters (C, kernel types if using SVC)
- Try different vectorization approaches (character n-grams, word embeddings)
- Consider more advanced techniques like ensemble methods or neural networks

For automated experiments and result tracking, use the `src/evaluation/svm_baseline_main.py` script.